In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn import neighbors
import face_recognition as fr
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import recall_score, precision_score, f1_score


import joblib as jb
print("import done")

In [ ]:
df_cele = pd.read_csv("celebrities.csv")
df = pd.read_csv("celebrity_images.csv")

In [ ]:
df_cele.head(5)

In [ ]:
df.head(5)

In [ ]:
face_encodings = []
face_label = []
for label, image_path in zip(df['celeb'], df['image_path']):
    image = fr.load_image_file(image_path)
    face_locations = fr.face_locations(image)
    if len(face_locations) == 1:
        face_encoding = fr.face_encodings(image, known_face_locations=face_locations)[0]
        face_encodings.append(face_encoding)
        face_label.append(label)
    else:
        print(f"Skipping image {image_path} due to {len(face_locations)} faces detected.")

In [ ]:
# image = fr.load_image_file(r"dataset/Abhay_Deol/Abhay_Deol.124.jpg")
# face_locations = fr.face_locations(image)
# fr.face_encodings(image, known_face_locations=face_locations)[0]

In [ ]:
len(face_encodings), len(face_label)

In [ ]:
df_new = pd.DataFrame({'label': face_label, 'encoding': face_encodings})

In [ ]:
df_new['label'].value_counts()

In [ ]:
df_new['label'].value_counts().reset_index()['count'].describe()

In [ ]:
df_new

In [ ]:
X = np.array(df_new['encoding'].tolist())
y = np.array(df_new['label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
knn = neighbors.KNeighborsClassifier(
    n_neighbors=3,
    algorithm="ball_tree",
    weights="distance"
)

knn.fit(X_train, y_train)
pred = knn.predict(X_test)

print(classification_report(y_test, pred, zero_division=0))
print("Accuracy:", accuracy_score(y_test, pred))

In [ ]:
new_image_path = "dataset/image copy 7.png"
def recognize_face(new_image_path, knn):
    image = fr.load_image_file(new_image_path)
    face_locations = fr.face_locations(image)
    if len(face_locations) == 1:
        face_encoding = fr.face_encodings(image, known_face_locations=face_locations)[0]
        if face_encoding is not None:
            pred = knn.predict([face_encoding])
            return pred[0]
        else:
            return None
    else:
        return None

In [ ]:
recognize_face("dataset/image copy 7.png", knn)

In [ ]:
jb.dump(knn, "knn_model.pkl")
